# CS4765/6765 Assignment 1: $N$-gram Language Models

**Due 1 October**

In this assignment you will implement a bigram language model and an interpolated language model. You will evaluate these models using perplexity and compare them to a unigram language model (which I've implemented for you).

Read through this notebook in its entirety before getting started. You should only make changes / write code in parts of the notebook where the instructions ask you to do so. These are indicated with TODO throughout.

Note that NLTK is used to access the data that we will use to train and evaluate our language models in this assignment. However, for the code that you write for this assignment, you must not use NLTK or any other NLP toolkits. You also must not use generative AI to complete this assignment.

The assignment must be able to run on Python 3.12 in the nlpfall2026venv virtual environment on the NLP VM available on the FCS lab machines.





## Data

We will use the Brown Corpus for this assignment. The Brown Corpus is a collection of roughly 1 million words of American English sampled from a wide range of sources. It is available through NLTK, which is available in the NLP VM on the lab machines.

The code cells below preprocess the data for us to use for language modelling.

In [4]:
import nltk, random
nltk.download('brown')
brown_sents = list(nltk.corpus.brown.sents())

# Seed the random number generator so that the random shuffling is the same each run
random.seed(1)
random.shuffle(brown_sents)

# Randomly shuffle the sentences in the Brown Corpus. The first 80% are for training. The next
# 10 % for dev, and the last 10% for test.
split_index_1 = int(len(brown_sents) * 0.8)
split_index_2 = int(len(brown_sents) * 0.9)
train_sents = brown_sents[:split_index_1]
dev_sents = brown_sents[split_index_1:split_index_2]
test_sents = brown_sents[split_index_2:]

# A sanity check
assert len(brown_sents) == len(train_sents) + len(dev_sents) + len(test_sents)

[nltk_data] Downloading package brown to /Users/isaac/nltk_data...
[nltk_data]   Package brown is already up-to-date!


In [5]:
# train_sents is a list. Each element of this list represents a sentence. Each sentence is
# itself a list consisting of the tokens in that sentence. Here is an example:
train_sents[4452]

['Financing', 'emerged', 'as', 'the', 'main', 'obstacle', '.']

In [6]:
# One challenge in NLP is how to handle words/tokens that do not occur in the training data.
# For this assignment, we  preprocess the data so that this does not occur. Specifically, we
# replace all hapax legomena (words that occur exactly once) in the training data by a special 
# token, <UNK>, which represents unkown words. For data that is used for evaluation, any word/token
# that does not occur in the training data is converted to <UNK>.
# (Note that in contemporary NLP, approaches to subword tokenization such as BPE address this issue,
# because an unseen token can be represented as a sequence of subwords that are known.)
UNK = "<UNK>"
def normalize_train_sents(sents):
    # Count how many times each token occurs in the sents
    token_counts = {}
    for s in sents:
        for token in s:
            token_counts[token] = token_counts.get(token, 0) + 1

    # Find the hapaxes, i.e., words/tokens that only occur once
    hapaxes = set([x for x in token_counts if token_counts[x] == 1])
    
    return [[w if w not in hapaxes else UNK for w in s] for s in sents]

train_sents = normalize_train_sents(train_sents)

In [7]:
# Here is the same example as before; some tokens have been replaced by <UNK>
train_sents[4452]

['<UNK>', 'emerged', 'as', 'the', 'main', 'obstacle', '.']

In [8]:
# Normalize the dev and test data, as described above. dev_sents and test_sents both have
# the same format as train_sents, i.e., a list of sentences where each sentence is itself
# a list of tokens.
train_vocab = set()
for sent in train_sents:
    for token in sent:
        train_vocab.add(token)

dev_sents = [[w if w in train_vocab else UNK for w in s] for s in dev_sents]
test_sents = [[w if w in train_vocab else UNK for w in s] for s in test_sents]

In [9]:
# Augment each sentence with a beginning of sentence (<s>) and end of sentence (</s>) marker

BOS = "<s>"
EOS = "</s>"

for s in train_sents + dev_sents + test_sents: 
    assert BOS not in s
    assert EOS not in s

def pad_sents(sents):
    return [[BOS] + s + [EOS] for s in sents]

train_sents = pad_sents(train_sents)
dev_sents = pad_sents(dev_sents)
test_sents = pad_sents(test_sents)


In [10]:
# Here is our example sentence once more, with beginning and end of sentence markers
train_sents[4452]

['<s>', '<UNK>', 'emerged', 'as', 'the', 'main', 'obstacle', '.', '</s>']

## Models

### Unigram language model

The starter code provides a unigram language model. This model does not use any smoothing.

### Bigram language model (6 marks)

Implement a bigram language model with Laplace (add-1) smoothing, as discussed in class and 3.1 and 3.6 of the textbook.

### Interpolated language model (2 marks)

Implement a language model that interpolates the above unigram and bigram
language models:

\begin{equation}
P(w_n|w_{n-1}) = (1-\lambda) P(w_n|w_{n-1}) + \lambda P(w_n) \tag{1}
\end{equation}

In this model $P(w_n|w_{n-1})$ is the probability of $w_n$ from the bigram language model (with Laplace smoothing) and $P(w_n)$ is the probability of $w_n$ from the unigram language model (which doesn't use any smoothing). As part of the assignmnet you will select an appropriate value for $\lambda$. (Further details on this are below.)

In [11]:
import math
EPS = 0.0001

# An unsmoothed unigram language model. You should read this to understand how it works. 
# See examples below of how to use it.
class UnsmoothedUnigramLM:
    def __init__(self, sentences):
        self.counts = {}
        self.train(sentences)

    def train(self, sentences):
        for sent in sentences:
            # Note that here we are treating the first token, which is always the beginning
            # of sentence marker, <s>, the same as any other token, similar to the approach
            # taken in the examples in lecture.
            for token in sent:
                self.counts[token] = self.counts.get(token, 0) + 1
                
        # Computing this sum once during training, instead of every
        # time it's needed in log_prob, speeds things up
        self.num_instances = sum(self.counts.values())

    def log_prob(self, token):
        # Compute probabilities in log space to avoid underflow errors
        # (This can become an issue when we multiply together many
        # probabilities)
        # Note that token will always be in self.counts because of the normalization
        # steps we've previously applied. (Any token that is not in the training 
        # data is normalized to the unknown token, <UNK>
        return math.log(self.counts[token]) - math.log(self.num_instances)

    def check_probs(self):
        # Hint: Writing code to check whether the probabilities you
        # have computed form a valid probability distribution is very
        # helpful, particularly when you start incorporating smoothing
        # It can be a bit slow, however, especially for bigram language 
        # models, so you might want to turn these checks off once 
        # you're convinced things are working correctly.

        # Make sure the probability for each token is between 0 and 1
        for t in self.counts:
            assert 0 - EPS < math.exp(self.log_prob(t)) < 1 + EPS
        # Make sure that the sum of probabilities for all words is 1
        assert 1 - EPS < \
            sum([math.exp(self.log_prob(w)) for w in self.counts]) < \
            1 + EPS

    # Return the log prob of each token in each sentence
    def predict(self, sents):
        result_log_probs = []
        for sent in sents:
            # Ignore first token because it is always <s>, which is only there
            # to provide context for the bigram language model.
            for t in sent[1:]:
                result_log_probs.append(self.log_prob(t))
        return result_log_probs


In [ ]:
# TODO Implement BigramLM following the explanation of the Bigram Language Model above by 
# completing the constructor and log_prob methods. You are welcome to use additional methods
# as needed for your solution, but you should not change the signature for the constructor or 
# log_prob or check_probs methods because other parts of the starter code (which you shouldn't 
# modify) rely on these. You also should not modify the predict method.

class BigramLM:
    def __init__(self, fname):
        self.unigram_counts = {}
        self.bigram_counts = {}
        self.continuations = {}
        self.vocab = set()

        for sent in fname:
            self.vocab.update(sent)
            for w1, w2 in zip(sent, sent[1:]):
                self.unigram_counts[w1] = self.unigram_counts.get(w1, 0) + 1
                bigram = (w1, w2)
                self.bigram_counts[bigram] = self.bigram_counts.get(bigram, 0) + 1
                self.continuations.setdefault(w1, set()).add(w2)

        self.vocab_size = len(self.vocab)
        
    def log_prob(self, w1, w2):
        numerator = self.bigram_counts.get((w1, w2), 0) + 1
        denominator = self.unigram_counts.get(w1, 0) + self.vocab_size
        return math.log(numerator) - math.log(denominator)

    # Return the log prob of each token in each sentence
    def predict(self, sents):
        result_log_probs = []
        for sent in sents:
            tokens = sent
            for i in range(len(tokens))[:-1]:
                t1 = tokens[i]
                t2 = tokens[i+1]
                result_log_probs.append(self.log_prob(t1, t2))
        return result_log_probs

    def check_probs(self):
        for w1, context_count in self.unigram_counts.items():
            seen = self.continuations[w1]
            numerator_sum = sum(self.bigram_counts[(w1, w2)] + 1 for w2 in seen)
            numerator_sum += self.vocab_size - len(seen)
            prob_sum = numerator_sum / (context_count + self.vocab_size)
            assert 1 - EPS < prob_sum < 1 + EPS

In [ ]:
# TODO Implement InterpLM following the explanation of the Interpolated Language Model above by 
# completing the log_prob method. You are welcome to use additional methods
# as needed for your solution, but you should not change any other parts of the code, other
# than implementing check_probs (if you choose to do so).

class InterpLM:
    def __init__(self, sentences, alpha, unigram_lm, bigram_lm):
        # Sanity check: alpha must be between 0 and 1
        assert 0 <= alpha <= 1
        self.alpha = alpha
        self.unigram_lm = unigram_lm
        self.bigram_lm = bigram_lm

    def log_prob(self, w1, w2):
        bigram_prob = math.exp(self.bigram_lm.log_prob(w1, w2))
        unigram_prob = math.exp(self.unigram_lm.log_prob(w2))
        prob = (1 - self.alpha) * bigram_prob + self.alpha * unigram_prob
        return math.log(prob)

    def predict(self, sents):
        result_log_probs = []
        for sent in sents:
            tokens = sent
            for i in range(len(tokens))[:-1]:
                t1 = tokens[i]
                t2 = tokens[i+1]
                result_log_probs.append(self.log_prob(t1, t2))
        return result_log_probs

    def check_probs(self):
        self.unigram_lm.check_probs()
        self.bigram_lm.check_probs()
        assert 1 - EPS < (1 - self.alpha) + self.alpha < 1 + EPS

In [14]:
# Create and train the language models
unigram_lm = UnsmoothedUnigramLM(train_sents)
bigram_lm = BigramLM(train_sents)

# This is just an example to show how to use InterpLM
interp_lm = InterpLM(train_sents, 0.1, unigram_lm, bigram_lm)


In [15]:
# You can comment out these lines out to run faster...
unigram_lm.check_probs()
bigram_lm.check_probs()
interp_lm.check_probs()

In [16]:
# An implementation of perplexity that follows the definition given in
# Equation 3.42 of the textbook
def perplexity(log_probs):
    logP = sum(log_probs)
    N = float(len(log_probs))
    HW = (-1/N) * logP
    result = math.exp(HW)
    return result

In [17]:
# Evaluate the language models on the dev data
dev_unigram_logprobs = unigram_lm.predict(dev_sents)
dev_bigram_logprobs = bigram_lm.predict(dev_sents)
dev_interp_logprobs = interp_lm.predict(dev_sents)

print("Unigram:", perplexity(dev_unigram_logprobs))
print("Bigram:", perplexity(dev_bigram_logprobs))
print("Interpolated (lambda=0.1):", perplexity(dev_interp_logprobs))

Unigram: 804.6896576631818
Bigram: 1878.2050376378666
Interpolated (lambda=0.1): 981.5294079224137


The code above uses a single value for lambda for the interpolated LM (0.1). Your task here is to find a good value for lambda. We will discuss approaches for doing this in lecture. Importantly, you must only consider the development data (and crucially not the test data) when doing so.

In [18]:
# Tune lambda on the development data only.
lambda_results = []
for i in range(101):
    lambda_value = i / 100
    candidate_lm = InterpLM(train_sents, lambda_value, unigram_lm, bigram_lm)
    candidate_perplexity = perplexity(candidate_lm.predict(dev_sents))
    lambda_results.append((candidate_perplexity, lambda_value))

best_dev_perplexity, best_lambda = min(lambda_results)
print("Best lambda:", best_lambda)
print("Development perplexity:", best_dev_perplexity)

Best lambda: 0.65
Development perplexity: 665.6850733843866


In [19]:
# When you are done all of the parts above, evaluate the models on the test data

test_unigram_logprobs = unigram_lm.predict(test_sents)
test_bigram_logprobs = bigram_lm.predict(test_sents)

print("Unigram", perplexity(test_unigram_logprobs))
print("Bigram", perplexity(test_bigram_logprobs))

# Evaluate once on the test data using the lambda selected on development data.
best_interp_lm = InterpLM(train_sents, best_lambda, unigram_lm, bigram_lm)
test_interp_logprobs = best_interp_lm.predict(test_sents)
print("Interpolated (lambda={}):".format(best_lambda),
      perplexity(test_interp_logprobs))


Unigram 807.6392797292832
Bigram 1897.7490600792125
Interpolated (lambda=0.65): 667.3581250301668


## Report (2 marks)

Write a brief report addressing the following points:

1. In my sample solution I found that the unigram model was better (i.e., had lower perplexity) than the bigram model, which is counter to what would be expected given the additional context available to the bigram model. What limitation of the bigram model that we implemented could have led to this? Can you suggest a way to improve the bigram model? (You don't actually have to implement anything here, just describe what you could potentially do to address this limitation?)

1. Clearly describe the process that you used to select the best value for lambda for the interpolated LM. What value of lambda did you find to work best?


### 1 Bigram-model limitation

The add-one-smoothed bigram model can perform worse than the unigram model because add-one smoothing is too aggressive for a large, sparse bigram space. Most possible bigrams are unseen. Adding one count to every possible continuation allocates a large amount of probability mass to those unseen events and removes too much mass from bigrams that were actually observed. A better model could use a more effective smoothing method, such as absolute discounting with interpolation, Katz backoff, or Kneser--Ney smoothing. These approaches discount observed counts more carefully and use lower-order probabilities when the bigram evidence is weak.

### 2 Selecting lambda

I selected lambda using only the development set. I evaluated the interpolated model at every value from 0.00 through 1.00 in increments of 0.01, computed development-set perplexity for each value, and chose the value with the lowest perplexity. The best value was **lambda = 0.65**, with a development perplexity of approximately **665.69**. I then fixed lambda at 0.65 for the single final evaluation on the test set.

## What to submit



## Grading

Your assignments will be graded based primarily on the correctness of the implementation and the written answers in the report.

Assignments that do not conform to the specifications outlined above might not be graded (e.g., modifying parts of the starter code that you were not asked to modify). Assignments that we are unable to run in a reasonable amount of time (less than five minutes) also might not be graded. Grades will be out of 10 and broken down as follows:

Bigram LM: 6

Interpolated LM: 2

Report : 2